# NB3 v2 — Vulnerabilidade Econômica Portuária (ComexStat + CUCI)

**Mapeia a exposição econômica das cadeias produtivas brasileiras a disrupções portuárias.**  
Responde à pergunta de pesquisa 4:  
> *Quais cadeias produtivas estão mais expostas a disrupções portuárias?*

### Abordagem
Análise **descritiva** de vulnerabilidade econômica focada em perfis de concentração
e exposição por cadeia produtiva. Sem teste de hipótese — apenas mapeamento factual
de "se este porto parar, o que está em risco."

### Classificação de produtos
Usa a **CUCI (Classificação Uniforme para o Comércio Internacional)** no nível **Grupo (3 dígitos)**,
que fornece ~260 grupos com granularidade adequada (soja ≠ café ≠ minério).
É o nível adotado oficialmente pela SECEX nas divulgações semanais da balança comercial.

### Inputs
| Fonte | Arquivo | Origem |
|-------|---------|--------|
| NB2 | `anomalias_classificadas.parquet` | Anomalias com classificação |
| NB2 | `ranking_vulnerabilidade.csv` | Ranking dos 20 portos |
| ComexStat | `EXP_YYYY.csv` | Exportações anuais (download MDIC) |
| MDIC | `NCM.csv` | Correlação NCM → CUCI item |
| MDIC | `NCM_CUCI.csv` | Hierarquia CUCI completa |
| Manual | `de_para_urf_porto.csv` | Mapeamento URF → porto |

### Outputs → `data/output/`
| Arquivo | Conteúdo |
|---------|----------|
| `comex_v2_perfil_cuci_porto.csv` | Composição CUCI por porto |
| `comex_v2_hhi_porto.csv` | HHI de concentração da pauta por porto |
| `comex_v2_hhi_cadeia.csv` | HHI de concentração portuária por cadeia CUCI |
| `comex_v2_matriz_vulnerabilidade.csv` | Cruzamento porto × cadeia |
| `comex_v2_ranking_portos.csv` | Ranking de vulnerabilidade dos portos |
| `comex_v2_ranking_cadeias.csv` | Ranking de cadeias em risco |

### Limitação: 16 de 18 portos
Dois portos do pipeline (Porto Alegre e Barra do Riacho) não possuem URF própria no ComexStat.
Angra dos Reis e São Sebastião foram excluídos do pipeline na v8 (dados insuficientes de Longo Curso).
A análise cobre **16 portos** com cobertura quase total do FOB marítimo.

## ═══ SEÇÃO A: Preparação de Dados ═══

In [ ]:
# A1 — Setup e carga de dados NB2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys, warnings, urllib.request, os
warnings.filterwarnings("ignore", category=FutureWarning)

sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import *

OUT = ROOT / "data" / "output"
AUX = ROOT / "data" / "aux"
AUX.mkdir(parents=True, exist_ok=True)
print(f"Config OK — {ROOT}")

# ── Outputs NB2 ──
anomalias = pd.read_parquet(OUT / "anomalias_classificadas.parquet")
vuln = pd.read_csv(OUT / "ranking_vulnerabilidade.csv")
print(f"Anomalias: {len(anomalias):,} linhas, {anomalias['porto'].nunique()} portos")
print(f"Ranking: {len(vuln)} portos")
print(f"Colunas anomalias: {list(anomalias.columns)}")

In [ ]:
# A2 — Carregar ComexStat
csv_files = sorted(COMEXSTAT_RAW.glob("EXP_*.csv"))
assert len(csv_files) > 0, f"ERRO: nenhum EXP_*.csv em {COMEXSTAT_RAW}"
print(f"Arquivos ComexStat: {len(csv_files)}")

frames = []
for f in csv_files:
    year = int(f.stem.split("_")[1])
    if year < MIN_DATA_YEAR:
        continue
    df = pd.read_csv(f, sep=";",
                     dtype={"CO_NCM": str, "CO_URF": str, "CO_PAIS": str,
                            "SG_UF_NCM": str, "CO_VIA": str,
                            "CO_ANO": int, "CO_MES": int})
    frames.append(df)
    print(f"  {f.name}: {len(df):>10,} linhas")

comex_raw = pd.concat(frames, ignore_index=True)
print(f"\nComexStat bruto: {len(comex_raw):,} linhas")

# Filtrar marítimo
comex = comex_raw[comex_raw["CO_VIA"] == "01"].copy()
print(f"Após filtro marítimo: {len(comex):,} linhas ({len(comex)/len(comex_raw):.0%})")

# De-para URF → Porto
urf_porto = pd.read_csv(DATA_RAW / "de_para_urf_porto.csv", dtype={"CO_URF": str})
comex["CO_URF"] = comex["CO_URF"].str.zfill(7)
urf_porto["CO_URF"] = urf_porto["CO_URF"].str.zfill(7)
comex = comex.merge(urf_porto[["CO_URF", "porto"]], on="CO_URF", how="inner")

# Filtrar apenas portos do pipeline NB2
portos_pipeline = set(vuln["porto"])
excluidos = set(comex["porto"].unique()) - portos_pipeline
comex = comex[comex["porto"].isin(portos_pipeline)]
if excluidos:
    print(f"Portos excluídos (fora do pipeline): {sorted(excluidos)}")

# NCM com 8 dígitos
comex["CO_NCM"] = comex["CO_NCM"].astype(str).str.zfill(8)

# Data mensal
comex["date"] = pd.to_datetime(
    comex["CO_ANO"].astype(str) + "-" +
    comex["CO_MES"].astype(str).str.zfill(2) + "-01")

print(f"\nComexStat final: {len(comex):,} linhas")
print(f"Portos: {comex['porto'].nunique()}, Período: {comex['date'].min().date()} — {comex['date'].max().date()}")

In [ ]:
# A3 — Download e merge NCM → CUCI Grupo

# Download tabelas CUCI do MDIC (uma vez)
urls = {
    "NCM.csv": "https://balanca.economia.gov.br/balanca/bd/tabelas/NCM.csv",
    "NCM_CUCI.csv": "https://balanca.economia.gov.br/balanca/bd/tabelas/NCM_CUCI.csv",
}
for name, url in urls.items():
    path = AUX / name
    if not path.exists():
        print(f"  Downloading {name}...")
        urllib.request.urlretrieve(url, path)
        print(f"  ✅ {name} ({path.stat().st_size / 1024:.0f} KB)")
    else:
        print(f"  {name} já existe ({path.stat().st_size / 1024:.0f} KB)")

# Ler tabelas de correlação
ncm_tab = pd.read_csv(AUX / "NCM.csv", sep=";", dtype=str, encoding="latin-1")
cuci_tab = pd.read_csv(AUX / "NCM_CUCI.csv", sep=";", dtype=str, encoding="latin-1")

# Limpar aspas
for tab in [ncm_tab, cuci_tab]:
    for col in tab.columns:
        tab[col] = tab[col].str.strip('"').str.strip()

print(f"NCM.csv: {len(ncm_tab):,} linhas, colunas: {list(ncm_tab.columns)}")
print(f"NCM_CUCI.csv: {len(cuci_tab):,} linhas, colunas: {list(cuci_tab.columns)}")

# Mapa NCM → CUCI item
ncm_cuci_map = ncm_tab[["CO_NCM", "CO_CUCI_ITEM"]].drop_duplicates()
ncm_cuci_map["CO_NCM"] = ncm_cuci_map["CO_NCM"].str.zfill(8)

# Mapa CUCI item → grupo (com nomes)
cuci_grupo = cuci_tab[["CO_CUCI_ITEM", "CO_CUCI_GRUPO", "NO_CUCI_GRUPO",
                        "CO_CUCI_SEC", "NO_CUCI_SEC"]].drop_duplicates()

# Join: NCM → CUCI_ITEM → CUCI_GRUPO
ncm_grupo = ncm_cuci_map.merge(cuci_grupo, on="CO_CUCI_ITEM", how="left")
print(f"\nMapa NCM→CUCI: {len(ncm_grupo):,} linhas")
print(f"  Com grupo CUCI: {ncm_grupo['CO_CUCI_GRUPO'].notna().sum():,}")
print(f"  Sem grupo CUCI: {ncm_grupo['CO_CUCI_GRUPO'].isna().sum():,}")

# Merge com ComexStat
n_before = len(comex)
comex = comex.merge(
    ncm_grupo[["CO_NCM", "CO_CUCI_GRUPO", "NO_CUCI_GRUPO",
               "CO_CUCI_SEC", "NO_CUCI_SEC"]].drop_duplicates(subset="CO_NCM"),
    on="CO_NCM", how="left")

cob = comex["CO_CUCI_GRUPO"].notna().mean()
cob_fob = comex.loc[comex["CO_CUCI_GRUPO"].notna(), "VL_FOB"].sum() / comex["VL_FOB"].sum()
print(f"\nCobertura CUCI: {cob:.1%} das linhas, {cob_fob:.1%} do FOB")
print(f"Grupos CUCI únicos: {comex['CO_CUCI_GRUPO'].nunique()}")

In [ ]:
# A5 — Mapeamento de países para regiões (destinos de exportação)
#
# Usado no NB4 para cruzar alertas regionais com fluxos de exportação.
# CO_PAIS é código de 3 dígitos do Siscomex.

# Baixar tabela de países do MDIC (se não existir)
pais_path = AUX / "NCM_PAIS.csv"
if not pais_path.exists():
    url_pais = "https://balanca.economia.gov.br/balanca/bd/tabelas/PAIS.csv"
    urllib.request.urlretrieve(url_pais, pais_path)
    print(f"  ✅ PAIS.csv ({pais_path.stat().st_size / 1024:.0f} KB)")

pais_tab = pd.read_csv(pais_path, sep=";", dtype=str, encoding="latin-1")
for col in pais_tab.columns:
    pais_tab[col] = pais_tab[col].str.strip('"').str.strip()

print(f"Tabela de países: {len(pais_tab)} linhas")
print(f"Colunas: {list(pais_tab.columns)}")

# Mapeamento manual: NO_PAIS → região
# Os top 50+ destinos de exportação BR mapeados manualmente
PAIS_REGIAO = {
    # Asia
    "China": "Asia", "Japão": "Asia", "Japan": "Asia",
    "Coreia do Sul": "Asia", "Coréia do Sul": "Asia",
    "Cingapura": "Asia", "Singapura": "Asia",
    "Índia": "Asia", "India": "Asia",
    "Tailândia": "Asia", "Malásia": "Asia",
    "Indonésia": "Asia", "Vietnã": "Asia",
    "Filipinas": "Asia", "Taiwan": "Asia",
    "Hong Kong": "Asia", "Bangladesh": "Asia",
    "Paquistão": "Asia",
    "Emirados Árabes Unidos": "Asia",
    "Arábia Saudita": "Asia",
    "Irã": "Asia", "Irã (República Islâmica)": "Asia",
    "Turquia": "Asia",
    # Europe
    "Países Baixos": "Europe", "Holanda": "Europe",
    "Bélgica": "Europe", "Alemanha": "Europe",
    "Espanha": "Europe", "França": "Europe",
    "Reino Unido": "Europe", "Itália": "Europe",
    "Portugal": "Europe", "Grécia": "Europe",
    "Polônia": "Europe", "Suécia": "Europe",
    "Dinamarca": "Europe", "Noruega": "Europe",
    "Finlândia": "Europe", "Romênia": "Europe",
    "Irlanda": "Europe", "Rússia": "Europe",
    "Federação da Rússia": "Europe",
    # Americas (excl Brasil)
    "Estados Unidos": "Americas", "Canadá": "Americas",
    "México": "Americas", "Argentina": "Americas",
    "Chile": "Americas", "Colômbia": "Americas",
    "Peru": "Americas", "Paraguai": "Americas",
    "Uruguai": "Americas", "Venezuela": "Americas",
    "Equador": "Americas", "Cuba": "Americas",
    "República Dominicana": "Americas", "Panamá": "Americas",
    # Africa
    "África do Sul": "Africa", "Egito": "Africa",
    "Marrocos": "Africa", "Nigéria": "Africa",
    "Argélia": "Africa", "Gana": "Africa",
    "Quênia": "Africa", "Angola": "Africa",
    "Moçambique": "Africa", "Senegal": "Africa",
    "Tunísia": "Africa", "Tanzânia": "Africa",
}

def map_regiao(co_pais, pais_nomes):
    """Mapeia CO_PAIS para região via nome."""
    nome = pais_nomes.get(co_pais, "")
    if not nome:
        return "Other"
    if nome in PAIS_REGIAO:
        return PAIS_REGIAO[nome]
    for pais_key, regiao in PAIS_REGIAO.items():
        if pais_key.lower() in nome.lower() or nome.lower() in pais_key.lower():
            return regiao
    return "Other"

# Criar dicionário CO_PAIS → NO_PAIS
comex_pais = comex[["CO_PAIS"]].drop_duplicates().merge(
    pais_tab[["CO_PAIS", "NO_PAIS"]].drop_duplicates(),
    on="CO_PAIS", how="left")
pais_nomes_dict = dict(zip(comex_pais["CO_PAIS"], comex_pais["NO_PAIS"].fillna("")))

# Mapear
comex["regiao_destino"] = comex["CO_PAIS"].apply(
    lambda x: map_regiao(x, pais_nomes_dict))

# Diagnóstico
print(f"\nCobertura de regiões:")
cob_regiao = comex.groupby("regiao_destino")["VL_FOB"].sum()
cob_total = cob_regiao.sum()
for reg in ["Asia", "Europe", "Americas", "Africa", "Other"]:
    fob = cob_regiao.get(reg, 0)
    print(f"  {reg:10s}: US$ {fob/1e9:.1f} bi ({fob/cob_total:.1%})")

# Top países não mapeados
other = comex[comex["regiao_destino"] == "Other"]
if len(other) > 0:
    top_other = (other.groupby("CO_PAIS")["VL_FOB"].sum()
                 .nlargest(10).reset_index())
    top_other = top_other.merge(
        comex_pais[["CO_PAIS", "NO_PAIS"]].drop_duplicates(),
        on="CO_PAIS", how="left")
    print(f"\n⚠ Top 10 países não mapeados (classificados como 'Other'):")
    for _, r in top_other.iterrows():
        print(f"  {r['CO_PAIS']} {r['NO_PAIS']:30s} US$ {r['VL_FOB']/1e9:.1f} bi")
    print("  → Adicionar ao dicionário PAIS_REGIAO se relevantes")

In [ ]:
# A4 — Preparar anomalias em frequência mensal
anomalias["date"] = pd.to_datetime(anomalias["date"])
anomalias["ym"] = anomalias["date"].dt.to_period("M").dt.to_timestamp()

anom_mensal = (anomalias.groupby(["porto", "ym"])
    .agg(n_anomalias=("date", "size"),
         classificacao=("classificacao", lambda x: x.mode().iloc[0]),
         dims=("dim", lambda x: ",".join(sorted(set(x)))))
    .reset_index())

anom_por_porto = anomalias.groupby("porto").agg(
    n_anomalias=("date", "size"),
    n_global=("classificacao", lambda x: (x == "global").sum()),
    n_nacional=("classificacao", lambda x: (x == "nacional").sum()),
    n_isolado=("classificacao", lambda x: (x == "isolado").sum()),
).reset_index()

print(f"Anomalias mensais: {len(anom_mensal):,} pares porto×mês")
print(f"Portos com anomalias: {anom_por_porto['porto'].nunique()}")
print(f"\nResumo por classificação:")
print(f"  Total: {len(anomalias):,}")
for cls in ["global", "nacional", "isolado"]:
    n = (anomalias["classificacao"] == cls).sum()
    print(f"  {cls}: {n:,} ({n/len(anomalias):.1%})")

## ═══ SEÇÃO B: Perfil Exportador dos Portos ═══

In [ ]:
# B1 — Composição da pauta por porto (CUCI grupo)

# Agregar: porto × CUCI grupo → FOB total
porto_cuci = (comex[comex["CO_CUCI_GRUPO"].notna()]
    .groupby(["porto", "CO_CUCI_GRUPO", "NO_CUCI_GRUPO"])
    .agg(fob=("VL_FOB", "sum"), kg=("KG_LIQUIDO", "sum"))
    .reset_index()
    .sort_values(["porto", "fob"], ascending=[True, False]))

# Participação % por porto
total_porto = porto_cuci.groupby("porto")["fob"].transform("sum")
porto_cuci["pct"] = porto_cuci["fob"] / total_porto

# Top 5 por porto
print("COMPOSIÇÃO CUCI POR PORTO (Top 5 grupos)")
print("=" * 80)
for porto in sorted(porto_cuci["porto"].unique()):
    sub = porto_cuci[porto_cuci["porto"] == porto].head(5)
    fob_bi = sub["fob"].sum() / 1e9
    cr5 = sub["pct"].sum()
    total_bi = total_porto[porto_cuci["porto"] == porto].iloc[0] / 1e9
    print(f"\n  {porto} — FOB total: US$ {total_bi:.1f} bi, CR5: {cr5:.0%}")
    for _, r in sub.iterrows():
        print(f"    {r['CO_CUCI_GRUPO']} {r['NO_CUCI_GRUPO'][:45]:45s} "
              f"US$ {r['fob']/1e9:.1f}bi ({r['pct']:.0%})")

In [ ]:
# B1b — Heatmap: Porto × top 15 CUCI grupos

# Top 15 grupos CUCI globais
top_cuci = (comex[comex["CO_CUCI_GRUPO"].notna()]
    .groupby("CO_CUCI_GRUPO")["VL_FOB"].sum()
    .nlargest(15).index.tolist())

# Nomes curtos
cuci_names = (porto_cuci[["CO_CUCI_GRUPO", "NO_CUCI_GRUPO"]]
    .drop_duplicates().set_index("CO_CUCI_GRUPO")["NO_CUCI_GRUPO"]
    .str[:35].to_dict())

# Pivot normalizado por porto
heat_data = (porto_cuci[porto_cuci["CO_CUCI_GRUPO"].isin(top_cuci)]
    .pivot_table(index="porto", columns="CO_CUCI_GRUPO",
                 values="pct", fill_value=0))

# Ordenar portos por FOB total
porto_order = (comex.groupby("porto")["VL_FOB"].sum()
    .sort_values(ascending=False).index)
heat_data = heat_data.reindex(porto_order)

# Labels com nome
col_labels = [f"{c} {cuci_names.get(c, '')[:25]}" for c in heat_data.columns]

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(heat_data * 100, annot=True, fmt=".0f", cmap="YlOrRd",
            ax=ax, linewidths=0.5, linecolor="white",
            xticklabels=col_labels,
            cbar_kws={"label": "% do FOB do porto"})
ax.set_title("Composição exportadora por porto — Top 15 CUCI Grupos")
ax.set_xlabel("Grupo CUCI")
ax.set_ylabel("Porto (ordenado por FOB total)")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / "comex_v2_heatmap_porto_cuci.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# B2 — HHI de concentração da pauta exportadora por porto

def hhi(series):
    """Herfindahl-Hirschman Index."""
    shares = series / series.sum()
    return (shares ** 2).sum()

hhi_porto = (comex[comex["CO_CUCI_GRUPO"].notna()]
    .groupby(["porto", "CO_CUCI_GRUPO"])["VL_FOB"].sum()
    .groupby("porto").apply(hhi)
    .rename("hhi_pauta")
    .sort_values(ascending=False)
    .reset_index())

# Adicionar FOB total
fob_total = comex.groupby("porto")["VL_FOB"].sum().rename("fob_total")
hhi_porto = hhi_porto.merge(fob_total, on="porto")

# Adicionar top CUCI
top1 = (porto_cuci.groupby("porto").first()
    [["CO_CUCI_GRUPO", "NO_CUCI_GRUPO", "pct"]]
    .rename(columns={"CO_CUCI_GRUPO": "top1_cuci",
                     "NO_CUCI_GRUPO": "top1_nome",
                     "pct": "top1_pct"}))
hhi_porto = hhi_porto.merge(top1, on="porto")

print("HHI DA PAUTA EXPORTADORA POR PORTO")
print("=" * 80)
print("  HHI > 0.25 = concentrado, 0.15-0.25 = moderado, < 0.15 = diversificado")
print()
for _, r in hhi_porto.iterrows():
    flag = "🔴" if r["hhi_pauta"] > 0.25 else "🟡" if r["hhi_pauta"] > 0.15 else "🟢"
    print(f"  {flag} {r['porto']:22s} HHI={r['hhi_pauta']:.3f}  "
          f"FOB={r['fob_total']/1e9:.1f}bi  "
          f"Top: {r['top1_cuci']} {r['top1_nome'][:30]} ({r['top1_pct']:.0%})")

In [ ]:
# B3 — Concentração reversa: dependência da cadeia no porto

# Para cada grupo CUCI: como o FOB se distribui entre portos?
cadeia_porto = (comex[comex["CO_CUCI_GRUPO"].notna()]
    .groupby(["CO_CUCI_GRUPO", "NO_CUCI_GRUPO", "porto"])
    ["VL_FOB"].sum().reset_index())

total_cadeia = cadeia_porto.groupby("CO_CUCI_GRUPO")["VL_FOB"].transform("sum")
cadeia_porto["pct_porto"] = cadeia_porto["VL_FOB"] / total_cadeia

# HHI por cadeia (concentração portuária)
hhi_cadeia = (cadeia_porto.groupby(["CO_CUCI_GRUPO", "NO_CUCI_GRUPO"])
    .apply(lambda g: hhi(g["VL_FOB"]), include_groups=False)
    .rename("hhi_portuario")
    .reset_index()
    .sort_values("hhi_portuario", ascending=False))

# FOB total por cadeia
fob_cadeia = comex[comex["CO_CUCI_GRUPO"].notna()].groupby("CO_CUCI_GRUPO")["VL_FOB"].sum()
hhi_cadeia = hhi_cadeia.merge(fob_cadeia.rename("fob_total"), on="CO_CUCI_GRUPO")

# Porto principal por cadeia
porto_principal = (cadeia_porto.sort_values("VL_FOB", ascending=False)
    .groupby("CO_CUCI_GRUPO").first()
    [["porto", "pct_porto"]]
    .rename(columns={"porto": "porto_principal", "pct_porto": "pct_principal"}))
hhi_cadeia = hhi_cadeia.merge(porto_principal, on="CO_CUCI_GRUPO")

# Top 30 cadeias por FOB (relevantes)
top_cadeias = hhi_cadeia.nlargest(30, "fob_total")

print("CONCENTRAÇÃO PORTUÁRIA POR CADEIA CUCI (Top 30 por FOB)")
print("=" * 90)
print("  HHI alto = cadeia depende de poucos portos = vulnerável a disrupções\n")
for _, r in top_cadeias.iterrows():
    flag = "🔴" if r["hhi_portuario"] > 0.25 else "🟡" if r["hhi_portuario"] > 0.15 else "🟢"
    print(f"  {flag} {r['CO_CUCI_GRUPO']} {r['NO_CUCI_GRUPO'][:40]:40s} "
          f"HHI={r['hhi_portuario']:.3f}  FOB={r['fob_total']/1e9:.1f}bi  "
          f"Principal: {r['porto_principal']} ({r['pct_principal']:.0%})")

In [ ]:
# B4 — Scatter: vulnerabilidade cruzada (HHI porto × HHI cadeia)

# Para cada par (porto, cadeia CUCI): posição no espaço de vulnerabilidade
# Eixo X: HHI do porto (concentração da pauta)
# Eixo Y: HHI da cadeia (dependência portuária)
# Tamanho: FOB do par

# Filtrar top 20 cadeias por FOB
top20_cuci = fob_cadeia.nlargest(20).index.tolist()

scatter_data = (cadeia_porto[cadeia_porto["CO_CUCI_GRUPO"].isin(top20_cuci)]
    .merge(hhi_porto[["porto", "hhi_pauta"]], on="porto")
    .merge(hhi_cadeia[["CO_CUCI_GRUPO", "hhi_portuario"]], on="CO_CUCI_GRUPO"))

# Apenas pares com FOB significativo (> 0.1% do total)
fob_min = comex["VL_FOB"].sum() * 0.001
scatter_data = scatter_data[scatter_data["VL_FOB"] > fob_min]

fig, ax = plt.subplots(figsize=(12, 10))
sizes = (scatter_data["VL_FOB"] / 1e9) * 3  # escala visual
sizes = sizes.clip(5, 300)

scatter = ax.scatter(scatter_data["hhi_pauta"],
                     scatter_data["hhi_portuario"],
                     s=sizes, alpha=0.6, edgecolors="gray", linewidth=0.5)

# Anotar os maiores pares
for _, r in scatter_data.nlargest(15, "VL_FOB").iterrows():
    label = f"{r['porto'][:12]}\n{r['CO_CUCI_GRUPO']}"
    ax.annotate(label, (r["hhi_pauta"], r["hhi_portuario"]),
                fontsize=6, alpha=0.8, ha="center",
                textcoords="offset points", xytext=(0, 8))

# Quadrantes
ax.axvline(0.20, color="gray", linestyle="--", alpha=0.3)
ax.axhline(0.20, color="gray", linestyle="--", alpha=0.3)
ax.text(0.02, 0.95, "Porto diversificado\nCadeia concentrada",
        transform=ax.transAxes, fontsize=8, alpha=0.5, va="top")
ax.text(0.98, 0.95, "MÁXIMA\nVULNERABILIDADE",
        transform=ax.transAxes, fontsize=9, alpha=0.7, va="top", ha="right",
        color="red", fontweight="bold")
ax.text(0.02, 0.02, "MÍNIMA\nVULNERABILIDADE",
        transform=ax.transAxes, fontsize=9, alpha=0.7, color="green", fontweight="bold")
ax.text(0.98, 0.02, "Porto concentrado\nCadeia diversificada",
        transform=ax.transAxes, fontsize=8, alpha=0.5, ha="right")

ax.set_xlabel("HHI do porto (concentração da pauta exportadora)")
ax.set_ylabel("HHI da cadeia (dependência portuária)")
ax.set_title("Vulnerabilidade cruzada: Porto × Cadeia CUCI")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(FIGS / "comex_v2_scatter_vulnerabilidade.png", dpi=150, bbox_inches="tight")
plt.show()

# Quadrante de máxima vulnerabilidade
critico = scatter_data[(scatter_data["hhi_pauta"] > 0.20) &
                        (scatter_data["hhi_portuario"] > 0.20)]
print(f"\nPares no quadrante de MÁXIMA VULNERABILIDADE: {len(critico)}")
for _, r in critico.nlargest(10, "VL_FOB").iterrows():
    print(f"  {r['porto']:22s} × {r['CO_CUCI_GRUPO']} {r['NO_CUCI_GRUPO'][:35]:35s} "
          f"FOB={r['VL_FOB']/1e9:.1f}bi")

## ═══ SEÇÃO C: Perfil de Anomalias por Cadeia Produtiva ═══

In [ ]:
# C1 — Incidência de anomalias por grupo CUCI

# Para cada grupo CUCI, quais portos o exportam?
cuci_portos = (comex[comex["CO_CUCI_GRUPO"].notna()]
    .groupby(["CO_CUCI_GRUPO", "porto"])["VL_FOB"].sum()
    .reset_index()
    .rename(columns={"VL_FOB": "fob_par"}))

# Total mensal de portos×meses no período
n_meses_total = comex["date"].nunique()

# Anomalias mensais por porto
anom_set = set(zip(anom_mensal["porto"], anom_mensal["ym"]))

# Para cada grupo CUCI: contar meses em que ALGUM porto exportador teve anomalia
exposicao_cuci = []
for cuci_g in cuci_portos["CO_CUCI_GRUPO"].unique():
    portos_deste_cuci = cuci_portos[cuci_portos["CO_CUCI_GRUPO"] == cuci_g]
    portos_lista = portos_deste_cuci["porto"].unique()
    fob_total_g = portos_deste_cuci["fob_par"].sum()
    
    # Meses com anomalia nos portos que exportam este grupo
    meses_afetados = set()
    fob_ponderado = 0
    for _, row in portos_deste_cuci.iterrows():
        porto_meses = {ym for (p, ym) in anom_set if p == row["porto"]}
        meses_afetados |= porto_meses
        # FOB ponderado: soma do peso (pct do FOB) × meses com anomalia
        pct_porto = row["fob_par"] / fob_total_g if fob_total_g > 0 else 0
        fob_ponderado += pct_porto * len(porto_meses)
    
    exposicao_cuci.append({
        "CO_CUCI_GRUPO": cuci_g,
        "n_portos": len(portos_lista),
        "meses_com_anomalia": len(meses_afetados),
        "pct_meses_afetados": len(meses_afetados) / n_meses_total if n_meses_total > 0 else 0,
        "exposicao_ponderada": fob_ponderado / n_meses_total if n_meses_total > 0 else 0,
        "fob_total": fob_total_g,
    })

df_exposicao = pd.DataFrame(exposicao_cuci)

# Merge com nomes
nomes = (porto_cuci[["CO_CUCI_GRUPO", "NO_CUCI_GRUPO"]]
    .drop_duplicates().set_index("CO_CUCI_GRUPO")["NO_CUCI_GRUPO"])
df_exposicao = df_exposicao.merge(
    nomes.rename("NO_CUCI_GRUPO"), left_on="CO_CUCI_GRUPO", right_index=True, how="left")

# Top 25 cadeias mais expostas (ponderada por FOB)
top_exp = df_exposicao.nlargest(25, "exposicao_ponderada")

print("TOP 25 CADEIAS CUCI MAIS EXPOSTAS A ANOMALIAS PORTUÁRIAS")
print("=" * 85)
print("  Exposição ponderada = Σ(peso_porto × meses_anomalia) / meses_total")
print("  Quanto maior, mais a cadeia depende de portos que tiveram anomalias\n")
for _, r in top_exp.iterrows():
    print(f"  {r['CO_CUCI_GRUPO']} {str(r.get('NO_CUCI_GRUPO',''))[:40]:40s} "
          f"exp={r['exposicao_ponderada']:.2f}  "
          f"{r['pct_meses_afetados']:.0%} meses  "
          f"FOB={r['fob_total']/1e9:.1f}bi  "
          f"portos={r['n_portos']}")

In [ ]:
# C2 — Tipo de anomalia por cadeia

# Para cada par (cadeia, porto), contar anomalias por tipo
anom_tipo_porto = (anomalias.groupby(["porto", "classificacao"])
    .size().rename("n").reset_index())

# Cruzar com pares cadeia × porto
cadeia_tipo = cuci_portos.merge(anom_tipo_porto, on="porto", how="inner")

# Agregar por cadeia × tipo, ponderado pelo peso do porto
cadeia_tipo_agg = (cadeia_tipo.groupby(["CO_CUCI_GRUPO", "classificacao"])
    .agg(n_total=("n", "sum"),
         fob_total=("fob_par", "sum"))
    .reset_index())

# Pivot para stacked bar
# Top 20 cadeias por FOB
top20_g = fob_cadeia.nlargest(20).index.tolist()
pivot_tipo = (cadeia_tipo_agg[cadeia_tipo_agg["CO_CUCI_GRUPO"].isin(top20_g)]
    .pivot_table(index="CO_CUCI_GRUPO", columns="classificacao",
                 values="n_total", fill_value=0))

# Normalizar
pivot_pct = pivot_tipo.div(pivot_tipo.sum(axis=1), axis=0)

# Labels
labels = [f"{g} {nomes.get(g, '')[:30]}" for g in pivot_pct.index]

fig, ax = plt.subplots(figsize=(12, 8))
pivot_pct[["global", "nacional", "isolado"]].plot.barh(
    stacked=True, ax=ax, color=["#6baed6", "#fd8d3c", "#74c476"])
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Proporção")
ax.set_title("Tipo de anomalia nos portos por cadeia CUCI (top 20)")
ax.legend(title="Classificação", loc="lower right")
plt.tight_layout()
plt.savefig(FIGS / "comex_v2_tipo_anomalia_cadeia.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# C3 — Diversificação como proteção

# Merge HHI portuário da cadeia com exposição a anomalias
divers = hhi_cadeia.merge(
    df_exposicao[["CO_CUCI_GRUPO", "exposicao_ponderada", "n_portos",
                  "pct_meses_afetados"]],
    on="CO_CUCI_GRUPO")

# Filtrar cadeias relevantes (FOB > 1bi)
divers = divers[divers["fob_total"] > 1e9]

fig, ax = plt.subplots(figsize=(10, 8))
sizes = (divers["fob_total"] / 1e9) * 2
sizes = sizes.clip(5, 200)

scatter = ax.scatter(divers["hhi_portuario"],
                     divers["exposicao_ponderada"],
                     s=sizes, alpha=0.6, edgecolors="gray", linewidth=0.5,
                     c=divers["n_portos"], cmap="RdYlGn")

plt.colorbar(scatter, ax=ax, label="Nº de portos")

# Anotar top
for _, r in divers.nlargest(12, "fob_total").iterrows():
    nome = r.get("NO_CUCI_GRUPO", r["CO_CUCI_GRUPO"])
    if isinstance(nome, str):
        nome = nome[:25]
    ax.annotate(f"{r['CO_CUCI_GRUPO']} {nome}",
                (r["hhi_portuario"], r["exposicao_ponderada"]),
                fontsize=7, alpha=0.8,
                textcoords="offset points", xytext=(5, 5))

ax.set_xlabel("HHI portuário da cadeia (concentração)")
ax.set_ylabel("Exposição ponderada a anomalias")
ax.set_title("Diversificação vs Exposição a anomalias (cadeias com FOB > US$ 1bi)")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(FIGS / "comex_v2_diversificacao_protecao.png", dpi=150, bbox_inches="tight")
plt.show()

# Correlação
corr = divers[["hhi_portuario", "exposicao_ponderada"]].corr().iloc[0, 1]
print(f"\nCorrelação (Pearson) HHI portuário × Exposição: {corr:.3f}")
print("  Espera-se correlação NEGATIVA: cadeias mais concentradas em 1 porto")
print("  podem ter MENOS meses afetados (se aquele porto tem poucas anomalias),")
print("  ou MAIS exposição ponderada (se aquele porto tem muitas anomalias).")

## ═══ SEÇÃO D: Estudos de Caso Descritivos ═══

In [ ]:
# D1-D3 — Estudos de caso descritivos

def estudo_caso_cuci(nome, portos, meses_evento, meses_ref, comex_df, cadeia_porto_df):
    """Estudo de caso descritivo com CUCI e alternativas portuárias."""
    print(f"\nESTUDO DE CASO: {nome}")
    print("=" * 70)

    ev_dates = pd.to_datetime(meses_evento)
    ref_dates = pd.to_datetime(meses_ref)

    for porto in portos:
        ev = comex_df[(comex_df["porto"] == porto) & (comex_df["date"].isin(ev_dates))]
        ref = comex_df[(comex_df["porto"] == porto) & (comex_df["date"].isin(ref_dates))]

        fob_ev = ev["VL_FOB"].sum()
        fob_ref = ref["VL_FOB"].sum()
        var = (fob_ev - fob_ref) / fob_ref if fob_ref > 0 else np.nan

        print(f"\n  {porto}: FOB evento={fob_ev/1e9:.2f}bi, "
              f"ref={fob_ref/1e9:.2f}bi" +
              (f", Δ={var:+.1%}" if pd.notna(var) else ""))

        # Top 5 CUCI afetados
        if len(ev) > 0 and ev["CO_CUCI_GRUPO"].notna().any():
            ev_cuci = (ev[ev["CO_CUCI_GRUPO"].notna()]
                .groupby(["CO_CUCI_GRUPO", "NO_CUCI_GRUPO"])["VL_FOB"].sum()
                .nlargest(5).reset_index())
            print(f"    Cadeias CUCI expostas:")
            for _, c in ev_cuci.iterrows():
                # Alternativas: outros portos que exportam este grupo
                alts = cadeia_porto_df[
                    (cadeia_porto_df["CO_CUCI_GRUPO"] == c["CO_CUCI_GRUPO"]) &
                    (cadeia_porto_df["porto"] != porto)]
                alt_str = ", ".join(
                    f"{r['porto']}({r['pct_porto']:.0%})"
                    for _, r in alts.nlargest(3, "VL_FOB").iterrows())
                print(f"      {c['CO_CUCI_GRUPO']} {str(c['NO_CUCI_GRUPO'])[:35]:35s} "
                      f"US$ {c['VL_FOB']/1e6:.0f}M  "
                      f"Alt: {alt_str}")

# 1. Greve dos Caminhoneiros
estudo_caso_cuci(
    "Greve dos Caminhoneiros (mai-jun 2018)",
    ["Santos", "Paranaguá", "Rio Grande", "Itaguaí"],
    ["2018-05-01", "2018-06-01"],
    ["2017-05-01", "2017-06-01"],
    comex, cadeia_porto)

# 2. Enchentes RS
estudo_caso_cuci(
    "Enchentes RS (mai-jun 2024)",
    ["Rio Grande"],
    ["2024-05-01", "2024-06-01"],
    ["2023-05-01", "2023-06-01"],
    comex, cadeia_porto)

# 3. COVID
estudo_caso_cuci(
    "COVID-19 (mar-jun 2020)",
    ["Santos", "Rio de Janeiro", "Paranaguá", "Vitória"],
    ["2020-03-01", "2020-04-01", "2020-05-01", "2020-06-01"],
    ["2019-03-01", "2019-04-01", "2019-05-01", "2019-06-01"],
    comex, cadeia_porto)

## ═══ SEÇÃO E: Rankings e Síntese ═══

In [ ]:
# E1 — Ranking de vulnerabilidade integrado (portos)

ranking_porto = hhi_porto[["porto", "hhi_pauta", "fob_total"]].copy()

# Adicionar anomalias
ranking_porto = ranking_porto.merge(
    anom_por_porto[["porto", "n_anomalias", "n_global", "n_nacional"]],
    on="porto", how="left")
ranking_porto["n_anomalias"] = ranking_porto["n_anomalias"].fillna(0)

# Score composto: normalizar cada componente [0, 1] e combinar
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-10)

ranking_porto["norm_anomalias"] = minmax(ranking_porto["n_anomalias"])
ranking_porto["norm_hhi"] = minmax(ranking_porto["hhi_pauta"])
ranking_porto["norm_fob"] = minmax(ranking_porto["fob_total"])

# Score: pesos iguais para frequência de anomalias, concentração, e relevância
ranking_porto["score_vuln"] = (
    0.40 * ranking_porto["norm_anomalias"] +
    0.30 * ranking_porto["norm_hhi"] +
    0.30 * ranking_porto["norm_fob"])

ranking_porto = ranking_porto.sort_values("score_vuln", ascending=False)

print("RANKING DE VULNERABILIDADE DOS PORTOS")
print("=" * 80)
print(f"  Score = 0.40×anomalias + 0.30×HHI_pauta + 0.30×FOB")
print(f"  {'#':>3s}  {'Porto':22s} Score  Anom   HHI   FOB(bi)")
print(f"  {'─'*3}  {'─'*22} {'─'*5}  {'─'*4}  {'─'*5} {'─'*7}")
for i, (_, r) in enumerate(ranking_porto.iterrows(), 1):
    print(f"  {i:3d}  {r['porto']:22s} {r['score_vuln']:.3f}  "
          f"{r['n_anomalias']:4.0f}  {r['hhi_pauta']:.3f} "
          f"{r['fob_total']/1e9:7.1f}")

In [ ]:
# E2 — Ranking de cadeias em risco

ranking_cadeia = hhi_cadeia[["CO_CUCI_GRUPO", "NO_CUCI_GRUPO",
                             "hhi_portuario", "fob_total",
                             "porto_principal", "pct_principal"]].copy()

# Merge com exposição
ranking_cadeia = ranking_cadeia.merge(
    df_exposicao[["CO_CUCI_GRUPO", "exposicao_ponderada", "n_portos"]],
    on="CO_CUCI_GRUPO", how="left")

# Filtrar relevantes (FOB > 500M)
ranking_cadeia = ranking_cadeia[ranking_cadeia["fob_total"] > 5e8]

# Score composto
ranking_cadeia["norm_hhi"] = minmax(ranking_cadeia["hhi_portuario"])
ranking_cadeia["norm_exp"] = minmax(ranking_cadeia["exposicao_ponderada"])
ranking_cadeia["norm_fob"] = minmax(ranking_cadeia["fob_total"])

ranking_cadeia["score_risco"] = (
    0.35 * ranking_cadeia["norm_hhi"] +
    0.35 * ranking_cadeia["norm_exp"] +
    0.30 * ranking_cadeia["norm_fob"])

ranking_cadeia = ranking_cadeia.sort_values("score_risco", ascending=False)

print("RANKING DE CADEIAS CUCI EM RISCO")
print("=" * 95)
print(f"  Score = 0.35×HHI_portuário + 0.35×exposição + 0.30×FOB")
print(f"  {'#':>3s}  {'CUCI':4s} {'Nome':35s} Score  HHI   Exp   FOB(bi) Principal")
print(f"  {'─'*3}  {'─'*4} {'─'*35} {'─'*5} {'─'*5} {'─'*5} {'─'*7} {'─'*20}")
for i, (_, r) in enumerate(ranking_cadeia.head(25).iterrows(), 1):
    nome = str(r.get('NO_CUCI_GRUPO', ''))[:35]
    print(f"  {i:3d}  {r['CO_CUCI_GRUPO']:4s} {nome:35s} "
          f"{r['score_risco']:.3f} {r['hhi_portuario']:.3f} "
          f"{r['exposicao_ponderada']:.3f} {r['fob_total']/1e9:7.1f} "
          f"{r['porto_principal']} ({r['pct_principal']:.0%})")

In [ ]:
# E3 — Exportar resultados

porto_cuci.to_csv(OUT / "comex_v2_perfil_cuci_porto.csv", index=False)
hhi_porto.to_csv(OUT / "comex_v2_hhi_porto.csv", index=False)
hhi_cadeia.to_csv(OUT / "comex_v2_hhi_cadeia.csv", index=False)
df_exposicao.to_csv(OUT / "comex_v2_exposicao_cuci.csv", index=False)
ranking_porto.to_csv(OUT / "comex_v2_ranking_portos.csv", index=False)
ranking_cadeia.to_csv(OUT / "comex_v2_ranking_cadeias.csv", index=False)

# Matriz de vulnerabilidade cruzada
vuln_matrix = scatter_data[["porto", "CO_CUCI_GRUPO", "NO_CUCI_GRUPO",
                            "VL_FOB", "hhi_pauta", "hhi_portuario"]].copy()
vuln_matrix.to_csv(OUT / "comex_v2_matriz_vulnerabilidade.csv", index=False)

print("✅ Outputs NB3v2 exportados para data/output/:")
for f in sorted(OUT.glob("comex_v2_*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name}: {size:.0f} KB")

# Resumo final
print(f"\n{'='*70}")
print("SÍNTESE NB3 v2")
print(f"{'='*70}")
print(f"  Portos analisados: {comex['porto'].nunique()}")
print(f"  Grupos CUCI mapeados: {comex['CO_CUCI_GRUPO'].nunique()}")
print(f"  Cobertura CUCI: {cob_fob:.1%} do FOB")
print(f"  Porto mais vulnerável: {ranking_porto.iloc[0]['porto']} "
      f"(score={ranking_porto.iloc[0]['score_vuln']:.3f})")
print(f"  Cadeia mais em risco: {ranking_cadeia.iloc[0]['CO_CUCI_GRUPO']} "
      f"{str(ranking_cadeia.iloc[0].get('NO_CUCI_GRUPO',''))[:35]} "
      f"(score={ranking_cadeia.iloc[0]['score_risco']:.3f})")
print(f"  Figuras salvas: {len(list(FIGS.glob('comex_v2_*')))}")

## ═══ SEÇÃO F: FOB Exposto a Anomalias ═══

Abordagem complementar aos rankings das Seções B-E. Em vez de scores compostos
com pesos arbitrários, cruza **temporalmente** as anomalias detectadas no NB2
com os dados de exportação do ComexStat.

Métrica central: **FOB Exposto** = valor em US$ que transitava pelo porto nos
meses em que ele apresentava comportamento anômalo. Não é estimativa de perda
nem relação causal — é mapeamento factual de exposição: "durante disrupções
neste porto, quanto valia o que estava passando por ali."

In [ ]:
# F1 — FOB exposto: cruzamento temporal anomalias × ComexStat
#
# Para cada porto×mês com anomalia detectada no NB2, quanto FOB
# estava sendo exportado ali? Fatia por CUCI e por classificação.

# Garantir que comex tem coluna ym compatível
comex["ym"] = comex["date"]  # já é primeiro dia do mês

# Join: porto×mês com anomalia → FOB por CUCI naquele mês
fob_anomalia = (comex
    .merge(anom_mensal[["porto", "ym", "n_anomalias", "classificacao"]],
           on=["porto", "ym"], how="inner"))

print(f"Linhas ComexStat em meses com anomalia: {len(fob_anomalia):,}")
print(f"  de {len(comex):,} totais ({len(fob_anomalia)/len(comex):.1%})")
print(f"  Portos: {fob_anomalia['porto'].nunique()}")
print(f"  Meses: {fob_anomalia['ym'].nunique()}")

# Agregar por porto × CUCI × classificação
fob_exp = (fob_anomalia
    .groupby(["porto", "CO_CUCI_GRUPO", "NO_CUCI_GRUPO", "classificacao"])
    .agg(fob_exposto=("VL_FOB", "sum"),
         meses_expostos=("ym", "nunique"))
    .reset_index())

# FOB total por porto×CUCI (para calcular % exposto)
fob_total_par = (comex
    .groupby(["porto", "CO_CUCI_GRUPO"])["VL_FOB"]
    .sum().reset_index(name="fob_total"))

fob_exp = fob_exp.merge(fob_total_par, on=["porto", "CO_CUCI_GRUPO"])
fob_exp["pct_exposto"] = fob_exp["fob_exposto"] / fob_exp["fob_total"]

print(f"\nFOB exposto agregado: {len(fob_exp):,} pares (porto × CUCI × classificação)")
fob_total_exposto = fob_exp["fob_exposto"].sum()
fob_total_geral = comex["VL_FOB"].sum()
print(f"FOB total exposto: US$ {fob_total_exposto/1e9:.1f} bi "
      f"({fob_total_exposto/fob_total_geral:.1%} do FOB total)")

In [ ]:
# F2 — Ranking de portos por FOB exposto
ranking_porto_fob = (fob_exp
    .groupby("porto")
    .agg(fob_exposto=("fob_exposto", "sum"),
         n_cadeias=("CO_CUCI_GRUPO", "nunique"))
    .reset_index()
    .sort_values("fob_exposto", ascending=False))

# Adicionar FOB total e % exposto
fob_total_porto = comex.groupby("porto")["VL_FOB"].sum().reset_index(name="fob_total")
ranking_porto_fob = ranking_porto_fob.merge(fob_total_porto, on="porto")
ranking_porto_fob["pct_exposto"] = (ranking_porto_fob["fob_exposto"] / 
                                     ranking_porto_fob["fob_total"])

# Adicionar HHI como contexto (não como componente de ranking)
ranking_porto_fob = ranking_porto_fob.merge(
    hhi_porto[["porto", "hhi_pauta"]], on="porto", how="left")

# Adicionar split por classificação
split_cls = (fob_exp
    .groupby(["porto", "classificacao"])["fob_exposto"]
    .sum().unstack(fill_value=0)
    .rename(columns=lambda c: f"fob_{c}"))
ranking_porto_fob = ranking_porto_fob.merge(split_cls, on="porto", how="left")

print("RANKING DE PORTOS — FOB EXPOSTO A ANOMALIAS")
print("=" * 85)
print(f"  {'#':>3s}  {'Porto':22s} {'FOB Exposto':>12s} {'% Total':>8s} "
      f"{'Nacional':>10s} {'Global':>10s} {'HHI':>5s}")
print(f"  {'─'*3}  {'─'*22} {'─'*12} {'─'*8} {'─'*10} {'─'*10} {'─'*5}")
for i, (_, r) in enumerate(ranking_porto_fob.iterrows(), 1):
    fob_nac = r.get("fob_nacional", 0) / 1e9
    fob_glo = r.get("fob_global", 0) / 1e9
    print(f"  {i:3d}  {r['porto']:22s} US${r['fob_exposto']/1e9:8.1f}bi "
          f"{r['pct_exposto']:7.1%} "
          f"US${fob_nac:6.1f}bi US${fob_glo:6.1f}bi "
          f"{r['hhi_pauta']:.3f}")

In [ ]:
# F3 — Ranking de cadeias por FOB exposto
ranking_cadeia_fob = (fob_exp
    .groupby(["CO_CUCI_GRUPO", "NO_CUCI_GRUPO"])
    .agg(fob_exposto=("fob_exposto", "sum"),
         n_portos=("porto", "nunique"))
    .reset_index()
    .sort_values("fob_exposto", ascending=False))

# Adicionar HHI portuário como contexto
ranking_cadeia_fob = ranking_cadeia_fob.merge(
    hhi_cadeia[["CO_CUCI_GRUPO", "hhi_portuario", "porto_principal", "pct_principal"]],
    on="CO_CUCI_GRUPO", how="left")

print("RANKING DE CADEIAS CUCI — FOB EXPOSTO A ANOMALIAS (Top 25)")
print("=" * 100)
print(f"  {'#':>3s}  {'CUCI':5s} {'Nome':35s} {'FOB Exposto':>12s} "
      f"{'Portos':>6s} {'HHI':>5s} {'Principal':>20s}")
print(f"  {'─'*3}  {'─'*5} {'─'*35} {'─'*12} {'─'*6} {'─'*5} {'─'*20}")
for i, (_, r) in enumerate(ranking_cadeia_fob.head(25).iterrows(), 1):
    nome = str(r.get("NO_CUCI_GRUPO", ""))[:35]
    princ = f"{r.get('porto_principal','')} ({r.get('pct_principal',0):.0%})"
    print(f"  {i:3d}  {r['CO_CUCI_GRUPO']:5s} {nome:35s} "
          f"US${r['fob_exposto']/1e9:8.1f}bi "
          f"{r['n_portos']:6d} {r.get('hhi_portuario',0):.3f} {princ}")

In [ ]:
# F4 — FOB exposto por UF
fob_uf_anom = (fob_anomalia
    .groupby(["SG_UF_NCM", "classificacao"])
    .agg(fob_exposto=("VL_FOB", "sum"))
    .reset_index())

fob_uf_total = (comex
    .groupby("SG_UF_NCM")["VL_FOB"]
    .sum().reset_index(name="fob_total"))

fob_uf_ranking = (fob_uf_anom
    .groupby("SG_UF_NCM")["fob_exposto"]
    .sum().reset_index()
    .merge(fob_uf_total, on="SG_UF_NCM")
    .sort_values("fob_exposto", ascending=False))

fob_uf_ranking["pct_exposto"] = fob_uf_ranking["fob_exposto"] / fob_uf_ranking["fob_total"]

# Adicionar HHI portuário da UF (concentração de escoamento)
hhi_uf_calc = (comex.groupby(["SG_UF_NCM", "porto"])["VL_FOB"]
    .sum().reset_index()
    .groupby("SG_UF_NCM")
    .apply(lambda g: ((g["VL_FOB"] / g["VL_FOB"].sum()) ** 2).sum())
    .reset_index(name="hhi_portuario_uf"))

fob_uf_ranking = fob_uf_ranking.merge(hhi_uf_calc, on="SG_UF_NCM", how="left")

print("FOB EXPOSTO POR UF")
print("=" * 75)
for i, (_, r) in enumerate(fob_uf_ranking.iterrows(), 1):
    print(f"  {i:2d}  {r['SG_UF_NCM']:2s}  "
          f"FOB exposto: US$ {r['fob_exposto']/1e9:.1f}bi  "
          f"({r['pct_exposto']:.0%} do total)  "
          f"HHI portuário: {r['hhi_portuario_uf']:.3f}")

In [ ]:
# F5 — Scatter: FOB exposto × HHI portuário por cadeia
# Substitui o scatter HHI×HHI com métrica de impacto real

scatter_f = ranking_cadeia_fob[ranking_cadeia_fob["fob_exposto"] > 1e8].copy()

fig, ax = plt.subplots(figsize=(14, 10))
sc = ax.scatter(
    scatter_f["fob_exposto"] / 1e9,
    scatter_f["hhi_portuario"],
    s=scatter_f["fob_exposto"] / scatter_f["fob_exposto"].max() * 500 + 20,
    c=scatter_f["hhi_portuario"],
    cmap="RdYlGn_r",
    alpha=0.7,
    edgecolors="white",
    linewidth=0.5)

# Labels nos top 15
for _, r in scatter_f.nlargest(15, "fob_exposto").iterrows():
    nome = str(r.get("NO_CUCI_GRUPO", ""))[:25]
    ax.annotate(f"{r['CO_CUCI_GRUPO']} {nome}",
                (r["fob_exposto"] / 1e9, r["hhi_portuario"]),
                fontsize=7, alpha=0.8,
                xytext=(5, 5), textcoords="offset points")

# Linhas de referência
ax.axhline(0.50, color="red", linestyle="--", alpha=0.3, label="HHI = 0.50")
ax.axhline(0.25, color="orange", linestyle="--", alpha=0.3, label="HHI = 0.25")

ax.set_xlabel("FOB Exposto a Anomalias (US$ bi)")
ax.set_ylabel("HHI Portuário da Cadeia (concentração de escoamento)")
ax.set_title("Vulnerabilidade de Cadeias Produtivas:\nFOB em risco × Alternativas portuárias")
ax.legend(loc="upper left")
plt.colorbar(sc, ax=ax, label="HHI Portuário")
plt.tight_layout()
plt.savefig(FIGS / "comex_v2_scatter_fob_exposto_hhi.png", dpi=150, bbox_inches="tight")
plt.show()

# Quadrante de máxima vulnerabilidade
critico_fob = scatter_f[
    (scatter_f["fob_exposto"] > scatter_f["fob_exposto"].quantile(0.75)) &
    (scatter_f["hhi_portuario"] > 0.50)]
print(f"\nQuadrante CRÍTICO (FOB alto + HHI alto): {len(critico_fob)} cadeias")
for _, r in critico_fob.nlargest(10, "fob_exposto").iterrows():
    nome = str(r.get("NO_CUCI_GRUPO", ""))[:35]
    print(f"  {r['CO_CUCI_GRUPO']} {nome:35s} "
          f"FOB exposto: US${r['fob_exposto']/1e9:.1f}bi  "
          f"HHI: {r['hhi_portuario']:.3f}  "
          f"Principal: {r.get('porto_principal','')}")

In [ ]:
# F6 — Heatmap temporal: FOB exposto por porto × semestre
fob_anomalia["semestre"] = (fob_anomalia["ym"].dt.year.astype(str) + "-" +
    np.where(fob_anomalia["ym"].dt.month <= 6, "S1", "S2"))

heat_fob = (fob_anomalia
    .groupby(["porto", "semestre"])["VL_FOB"]
    .sum().reset_index()
    .pivot_table(index="porto", columns="semestre",
                 values="VL_FOB", fill_value=0))

# Filtrar período relevante (2016+, excluir 2015 parcial)
cols_keep = [c for c in heat_fob.columns if not c.startswith("2015")]
heat_fob = heat_fob[sorted(cols_keep)]

# Converter para bilhões
heat_fob_bi = heat_fob / 1e9

# Ordenar por FOB total exposto
porto_order = heat_fob.sum(axis=1).sort_values(ascending=False).index
heat_fob_bi = heat_fob_bi.reindex(porto_order)

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(heat_fob_bi, annot=True, fmt=".1f", cmap="YlOrRd",
            ax=ax, linewidths=0.5, linecolor="white",
            cbar_kws={"label": "FOB exposto (US$ bi)"})
ax.set_title("FOB em trânsito durante anomalias — por porto × semestre")
ax.set_ylabel("Porto (ordenado por FOB exposto total)")
ax.set_xlabel("Semestre")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / "comex_v2_heatmap_fob_exposto_temporal.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# F7 — Exportar outputs da Seção F
fob_exp.to_csv(OUT / "comex_v2_fob_exposto.csv", index=False)
ranking_porto_fob.to_csv(OUT / "comex_v2_ranking_portos_fob.csv", index=False)
ranking_cadeia_fob.to_csv(OUT / "comex_v2_ranking_cadeias_fob.csv", index=False)
fob_uf_ranking.to_csv(OUT / "comex_v2_fob_exposto_uf.csv", index=False)

print("✅ Outputs Seção F exportados:")
for f in sorted(OUT.glob("comex_v2_*fob*")):
    print(f"  {f.name}: {f.stat().st_size/1024:.0f} KB")

# Síntese comparativa
print(f"\n{'='*70}")
print("COMPARAÇÃO: SEÇÃO E (scores compostos) vs SEÇÃO F (FOB exposto)")
print(f"{'='*70}")
print(f"  Seção E — Porto mais vulnerável: {ranking_porto.iloc[0]['porto']} "
      f"(score={ranking_porto.iloc[0]['score_vuln']:.3f})")
print(f"  Seção F — Porto mais exposto:    {ranking_porto_fob.iloc[0]['porto']} "
      f"(US$ {ranking_porto_fob.iloc[0]['fob_exposto']/1e9:.1f}bi)")
print(f"\n  Seção E — Cadeia mais em risco:  {ranking_cadeia.iloc[0]['CO_CUCI_GRUPO']} "
      f"(score={ranking_cadeia.iloc[0]['score_risco']:.3f})")
print(f"  Seção F — Cadeia mais exposta:   {ranking_cadeia_fob.iloc[0]['CO_CUCI_GRUPO']} "
      f"(US$ {ranking_cadeia_fob.iloc[0]['fob_exposto']/1e9:.1f}bi)")

In [ ]:
# A6 — FOB por cadeia × porto × região de destino (último ano completo)
#
# Output: usado pelo NB4 para cruzar alertas regionais com cadeias.
# Fonte: ComexStat (CO_PAIS → região), processado na célula A5.
# Métrica: FOB em US$ (valor econômico da exposição).
# Período: último ano completo (evita acúmulo discricionário de vários anos).

FOB_ANO_REF = comex["CO_ANO"].max()  # último ano completo no dataset
comex_ref = comex[(comex["CO_ANO"] == FOB_ANO_REF) & (comex["regiao_destino"] != "Other")]
print(f"Ano de referência para FOB regional: {FOB_ANO_REF}")
print(f"  Registros no período: {len(comex_ref):,}")

fob_destino = (comex_ref
    .groupby(["porto", "CO_CUCI_GRUPO", "NO_CUCI_GRUPO", "regiao_destino"])
    .agg(fob_regional=("VL_FOB", "sum"))
    .reset_index())

# Adicionar HHI portuário da cadeia (contexto)
fob_destino = fob_destino.merge(
    hhi_cadeia[["CO_CUCI_GRUPO", "hhi_portuario", "porto_principal", "pct_principal"]],
    on="CO_CUCI_GRUPO", how="left")

# Agregar: cadeia × região (sem porto, para ranking)
fob_cadeia_regiao = (fob_destino
    .groupby(["CO_CUCI_GRUPO", "NO_CUCI_GRUPO", "regiao_destino"])
    .agg(fob_regional=("fob_regional", "sum"),
         n_portos=("porto", "nunique"))
    .reset_index()
    .merge(hhi_cadeia[["CO_CUCI_GRUPO", "hhi_portuario",
                        "porto_principal", "pct_principal"]],
           on="CO_CUCI_GRUPO", how="left"))

# Agregar: porto × região (para exposição regional)
fob_porto_regiao = (comex_ref
    .groupby(["porto", "regiao_destino"])
    .agg(fob_regional=("VL_FOB", "sum"))
    .reset_index())

# Participação % de cada região por porto
total_por_porto = fob_porto_regiao.groupby("porto")["fob_regional"].transform("sum")
fob_porto_regiao["pct_regiao"] = fob_porto_regiao["fob_regional"] / total_por_porto

# Diagnóstico
print(f"\nFOB POR PORTO × REGIÃO DE DESTINO — {FOB_ANO_REF} (top 5 portos por região)")
print("=" * 70)
for reg in ["Asia", "Europe", "Americas", "Africa"]:
    sub = fob_porto_regiao[fob_porto_regiao["regiao_destino"] == reg]
    sub = sub.nlargest(5, "fob_regional")
    print(f"\n  {reg}:")
    for _, r in sub.iterrows():
        print(f"    {r['porto']:25s} US$ {r['fob_regional']/1e9:.1f} bi "
              f"({r['pct_regiao']:.0%} da carga do porto)")

print(f"\nFOB POR CADEIA × REGIÃO — {FOB_ANO_REF} (top 5 cadeias por região)")
print("=" * 70)
for reg in ["Asia", "Europe", "Americas"]:
    sub = fob_cadeia_regiao[fob_cadeia_regiao["regiao_destino"] == reg]
    sub = sub.nlargest(5, "fob_regional")
    print(f"\n  {reg}:")
    for _, r in sub.iterrows():
        nome = str(r.get("NO_CUCI_GRUPO", ""))[:35]
        print(f"    {r['CO_CUCI_GRUPO']:5s} {nome:35s} "
              f"US$ {r['fob_regional']/1e9:.1f} bi  "
              f"HHI={r.get('hhi_portuario',0):.2f}")

# Exportar
fob_destino.to_csv(OUT / "comex_v2_fob_cadeia_porto_regiao.csv", index=False)
fob_cadeia_regiao.to_csv(OUT / "comex_v2_fob_cadeia_regiao.csv", index=False)
fob_porto_regiao.to_csv(OUT / "comex_v2_fob_porto_regiao.csv", index=False)

print(f"\n✅ Outputs de destino exportados (referência: {FOB_ANO_REF}):")
print(f"  comex_v2_fob_cadeia_porto_regiao.csv: {len(fob_destino):,} linhas")
print(f"  comex_v2_fob_cadeia_regiao.csv: {len(fob_cadeia_regiao):,} linhas")
print(f"  comex_v2_fob_porto_regiao.csv: {len(fob_porto_regiao):,} linhas")
